# PapyrusLab E00 — run `preflight`

Notebook generato da `scripts/build_e00_notebooks.py` (non modificare a mano). Piano congelato: `docs/plans/2026-09-06-e00-controllo-noto-w035.md`.
Ogni controllo di arresto del piano è un'asserzione: se fallisce, il run si ferma e il log dice dove.

- Output persistiti: `/kaggle/working/e00/out` e `/kaggle/working/e00/logs`
- File pesanti (codice, checkpoint, cache), non persistiti: `/tmp/e00`


In [ ]:
MODE = "preflight"
SEED = None          # None nel preflight
WORK = "/kaggle/working/e00"
HEAVY = "/tmp/e00"
ZARR = "https://vesuvius-challenge-open-data.s3.amazonaws.com/PHerc0139/segments/20260317000000-w035_2026031718/surface-volumes/9.362um-1.2m-113keV-volume-20250728140407.zarr"
TORCH_EXPECTED = "2.10.0+cpu"   # immagine CPU di Kaggle: 2.10.0+cpu; immagine GPU: 2.10.0+cu128
print("MODE", MODE, "SEED", SEED, "torch atteso", TORCH_EXPECTED)


In [ ]:
%%bash
# Passo 1 — radice misurabile, cache e temporanei dirottati, guardia dei 10 GB
set -e
mkdir -p /kaggle/working/e00/out /kaggle/working/e00/logs /tmp/e00/tmp /tmp/e00/cache/pip /tmp/e00/cache/hf /tmp/e00/checkpoints /tmp/e00/labels
cat > /kaggle/working/e00/env.sh <<'EOF'
export WORK=/kaggle/working/e00
export HEAVY=/tmp/e00
export TMPDIR=$HEAVY/tmp PIP_CACHE_DIR=$HEAVY/cache/pip HF_HOME=$HEAVY/cache/hf
export ZARR="https://vesuvius-challenge-open-data.s3.amazonaws.com/PHerc0139/segments/20260317000000-w035_2026031718/surface-volumes/9.362um-1.2m-113keV-volume-20250728140407.zarr"
export LIMIT_BYTES=$((10*1024*1024*1024))
disk_check () {
  local used
  used=$(( $(du -sb "$WORK" | cut -f1) + $(du -sb "$HEAVY" | cut -f1) ))
  echo "spazio_byte=$used ($1)" | tee -a "$WORK/logs/disk_check.log"
  if [ "$used" -gt "$LIMIT_BYTES" ]; then echo "STOP: superati 10 GB ($1)" | tee -a "$WORK/logs/disk_check.log"; exit 1; fi
}
EOF
source /kaggle/working/e00/env.sh
echo "MODE=preflight SEED=None start=$(date -u +%FT%TZ)" > $WORK/logs/run_info.txt
disk_check "inizio"
df -h /kaggle/working /tmp | tail -2


In [ ]:
# Passo 1 (segue) — versioni dell'ambiente e rete verso le tre sorgenti
import sys, platform, json, urllib.request, torch
env = {"python": sys.version.split()[0], "platform": platform.platform(),
       "torch": torch.__version__, "cuda_available": torch.cuda.is_available(),
       "cuda_device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0}
print(json.dumps(env, indent=1))
json.dump(env, open(f"{WORK}/logs/env_before_install.json", "w"), indent=1)
assert env["torch"] == TORCH_EXPECTED, f"STOP: PyTorch inatteso {env['torch']} (atteso {TORCH_EXPECTED}): Kaggle ha cambiato immagine, aggiornare il dossier"
if MODE == "preflight":
    assert not env["cuda_available"], "STOP: il preflight deve girare senza acceleratore"
else:
    assert env["cuda_available"], "STOP: run GPU senza CUDA disponibile"
for url in [ZARR + "/0/.zarray",
            "https://huggingface.co/api/models/scrollprize/ink_9um",
            "https://huggingface.co/api/buckets/scrollprize/datasets"]:
    with urllib.request.urlopen(url, timeout=30) as r:
        print(r.status, url[:80]); assert r.status == 200, f"STOP: rete non raggiunge {url}"


In [ ]:
%%bash
# Passo 2 — checkout parziale di villa al commit congelato
set -e
source /kaggle/working/e00/env.sh
cd $HEAVY
[ -d villa/.git ] || git clone -q --filter=blob:none --no-checkout https://github.com/ScrollPrize/villa.git
cd villa
git sparse-checkout init --cone >/dev/null
git sparse-checkout set ink-detection vesuvius >/dev/null
git checkout -q 3ea17f54a9b3d5fd1aaf73e1d2c8386dbaa9f30e
HEAD=$(git rev-parse HEAD); echo "villa HEAD=$HEAD" | tee $WORK/logs/villa_commit.txt
[ "$HEAD" = "3ea17f54a9b3d5fd1aaf73e1d2c8386dbaa9f30e" ] || { echo "STOP: commit villa diverso"; exit 1; }
ls -l ink-detection/koine_machines/inference/infer.py vesuvius/pyproject.toml ink-detection/uv.lock
disk_check "dopo checkout"


In [ ]:
# Passo 3 — installazione sul Python di sistema, tutto con --no-deps, PyTorch intatto
import subprocess, re, json, sys
VILLA = f"{HEAVY}/villa"
PY = sys.executable

def sh(args):                      # argomenti come lista: nessuna interpretazione della shell
    return subprocess.run(args, capture_output=True, text=True)

def pip(*args):
    return sh([PY, "-m", "pip", "install", "--no-deps", *args])

def torch_version():
    return sh([PY, "-c", "import torch; print(torch.__version__)"]).stdout.strip()

torch_before = torch_version()
for pkg in ["vesuvius", "ink-detection"]:
    r = pip("-e", f"{VILLA}/{pkg}")
    assert r.returncode == 0, "STOP: pip install fallita\n" + r.stderr[-3000:]
r = pip("zarr==2.18.7", "numcodecs==0.15.1")   # il codice usa l'API Zarr v2
assert r.returncode == 0, "STOP: pip install zarr/numcodecs fallita\n" + r.stderr[-3000:]

lock = open(f"{VILLA}/ink-detection/uv.lock", encoding="utf-8").read()
def locked_version(dist):
    m = re.search(r'\[\[package\]\]\nname = "' + re.escape(dist.lower()) + r'"\nversion = "([^"]+)"', lock)
    return m.group(1) if m else None

ALIAS = {"cv2": "opencv-contrib-python-headless", "PIL": "pillow", "yaml": "pyyaml", "nrrd": "pynrrd",
         "cc3d": "connected-components-3d", "skimage": "scikit-image", "sklearn": "scikit-learn"}
added, numpy_downgraded = [], False

def ensure_import(modname):
    """Importa modname nel Python di sistema; per ogni modulo mancante installa (--no-deps) la versione del lock."""
    global numpy_downgraded
    for attempt in range(12):
        r = sh([PY, "-c", f"import {modname}"])
        if r.returncode == 0:
            return
        err = r.stderr
        m = re.search(r"No module named '([^'.]+)", err)
        if m:
            mod = m.group(1)
            assert re.fullmatch(r"[A-Za-z0-9_]+", mod), f"STOP: nome di modulo inatteso {mod!r}"
            cands = ([ALIAS[mod]] if mod in ALIAS else []) + [mod, mod.replace("_", "-")]
            dist = next((c for c in cands if locked_version(c)), None)
            assert dist, f"STOP: modulo mancante '{mod}' non presente in uv.lock:\n" + err[-2000:]
            ver = locked_version(dist)
            r2 = pip(f"{dist}=={ver}")
            assert r2.returncode == 0, f"STOP: pip install {dist}=={ver} fallita\n" + r2.stderr[-3000:]
            added.append({"module": mod, "dist": dist, "version": ver}); print("aggiunto", dist, ver, "per", modname)
        elif "numpy" in err.lower() and not numpy_downgraded:
            # unica eccezione ammessa dal piano: NumPy piu' recente di 2.2
            r2 = pip("numpy<=2.2"); assert r2.returncode == 0, r2.stderr[-3000:]
            numpy_downgraded = True; added.append({"module": "numpy", "dist": "numpy", "version": "<=2.2 (eccezione piano)"})
        else:
            raise AssertionError(f"STOP: import di {modname} fallito per motivo diverso da modulo mancante:\n" + err[-3000:])
    raise AssertionError(f"STOP: import di {modname} ancora fallito dopo i tentativi ammessi")

# Il run seed42 v1 (6 settembre 2026) e' fallito perche' koine_machines importa `vesuvius` solo quando costruisce
# il modello, e `vesuvius` richiede `nrrd`: l'import del solo entry point non basta. Si importano tutti i moduli
# realmente usati a run time.
for modname in ["koine_machines.inference.infer", "vesuvius", "vesuvius.models.build.build_network_from_config",
                "koine_machines.models.make_model"]:
    ensure_import(modname)
# Dipendenze caricate pigramente (non intercettabili con un import): il run seed42 v2 (6 settembre 2026) ha completato
# l'inferenza e si e' fermato scrivendo il TIFF perche' la compressione LZW di tifffile richiede `imagecodecs`.
for lazy in ["imagecodecs"]:
    if sh([PY, "-c", f"import {lazy}"]).returncode != 0:
        ver = locked_version(lazy); assert ver, f"STOP: {lazy} non presente in uv.lock"
        r2 = pip(f"{lazy}=={ver}"); assert r2.returncode == 0, f"STOP: pip install {lazy}=={ver} fallita\n" + r2.stderr[-3000:]
        added.append({"module": lazy, "dist": lazy, "version": ver}); print("aggiunto", lazy, ver, "(dipendenza pigra di tifffile)")
assert len(added) <= 10, f"STOP: {len(added)} pacchetti aggiunti, oltre il limite di dieci del piano"

torch_after = torch_version()
assert torch_after == torch_before == TORCH_EXPECTED, f"STOP: PyTorch cambiato da {torch_before} a {torch_after}"
helptxt = sh([PY, "-m", "koine_machines.inference.infer", "--help"]).stdout
for flag in ["--no-compile", "--gpus", "--layer-start"]:
    assert flag in helptxt, f"STOP: opzione {flag} assente nell'entry point"
hf = sh(["hf", "--version"])
if hf.returncode != 0:
    r2 = pip("huggingface_hub"); assert r2.returncode == 0, r2.stderr[-2000:]
    hf = sh(["hf", "--version"]); added.append({"module": "hf", "dist": "huggingface_hub", "version": hf.stdout.strip()})
assert sh(["hf", "buckets", "--help"]).returncode == 0, "STOP: 'hf buckets' non disponibile: huggingface_hub troppo vecchio"
pipl = [l for l in sh([PY, "-m", "pip", "list"]).stdout.splitlines()
        if re.match(r"(?i)^(torch|torchvision|zarr|numcodecs|numpy|tifffile|timm|scipy|fsspec|s3fs|aiohttp|huggingface.hub|monai|koine.machines|vesuvius|albumentations|einops|opencv) ", l)]
open(f"{WORK}/logs/pip_versions.txt", "w").write("\n".join(pipl) + "\n")
info = {"torch_before": torch_before, "torch_after": torch_after, "added_packages": added, "hf_version": hf.stdout.strip()}
json.dump(info, open(f"{WORK}/logs/install.json", "w"), indent=1)
print(json.dumps(info, indent=1)); print("\n".join(pipl))


In [ ]:
%%bash
# Passo 4 — checkpoint e label, con verifica esatta di hash e conteggi
set -e
source /kaggle/working/e00/env.sh
cd $HEAVY
disk_check "prima dei download"
hf download scrollprize/ink_9um hybrid_3d2d-seed42/step-075000.pth hybrid_3d2d-seed43/step-075000.pth \
   --revision 7109667e2607db1b90c37c8b09cb876ea7fe7bb1 --local-dir checkpoints/ink_9um > /dev/null
sha256sum checkpoints/ink_9um/hybrid_3d2d-seed42/step-075000.pth checkpoints/ink_9um/hybrid_3d2d-seed43/step-075000.pth | tee $WORK/logs/checkpoints_sha256.txt
stat -c "%s %n" checkpoints/ink_9um/hybrid_3d2d-seed4*/step-075000.pth | tee -a $WORK/logs/checkpoints_sha256.txt
grep -q "^e635558ae6a1a807a7e5ec1e83adfd45bc3c0ac53883ea43f1d4e085d62a9cab  checkpoints/ink_9um/hybrid_3d2d-seed42/step-075000.pth" $WORK/logs/checkpoints_sha256.txt || { echo "STOP: SHA-256 seed42 diverso"; exit 1; }
grep -q "^2aeaa85a35ef28d7bc7bf3e848c4a6a91385e9132710927fdba41133c4ecb28f  checkpoints/ink_9um/hybrid_3d2d-seed43/step-075000.pth" $WORK/logs/checkpoints_sha256.txt || { echo "STOP: SHA-256 seed43 diverso"; exit 1; }
disk_check "dopo i checkpoint"


In [ ]:
# Passo 4 (segue) — label w035 dal dataset Kaggle privato, verificata file per file contro manifest.json
# Kaggle estrae automaticamente w035_labels.tar al caricamento: il dataset espone w035_labels/w035/... e manifest.json
# (elenco dell'API del bucket con percorsi e dimensioni). La verifica e' su percorsi, dimensioni e conteggi.
import hashlib, os, shutil, json, glob
EXPECTED = (5128, 737833)
LABEL_TREE_SHA256 = "09037f1d0ccc008c5f619b2a4f41a554d1abf50739fd02469a9a2fb799731f27"
PREFIX = "ink_9um/labels/native9-scrollprizeorg-21slices/w035/"
LABEL_READY = False

def tree_sha256(root):
    """Stessa definizione di scripts/build_w035_label_dataset.py: sha256 su righe ordinate 'relpath\\nsha256(file)\\n'."""
    per_file = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            per_file[os.path.relpath(p, root).replace(os.sep, "/")] = hashlib.sha256(open(p, "rb").read()).hexdigest()
    h = hashlib.sha256()
    for rel, digest in sorted(per_file.items()):
        h.update(f"{rel}\n{digest}\n".encode())
    return h.hexdigest(), per_file

def check_label_tree(dest, source_desc):
    n = sum(len(fs) for _, _, fs in os.walk(dest)); b = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(dest) for f in fs)
    assert (n, b) == EXPECTED, f"STOP: label con {n} file / {b} byte, attesi {EXPECTED}"
    tsha, _ = tree_sha256(dest)
    assert tsha == LABEL_TREE_SHA256, f"STOP: contenuto della label diverso da quello congelato (tree sha256 {tsha})"
    open(f"{WORK}/logs/label_count.txt", "w").write(f"file={n} byte={b} tree_sha256={tsha} source={source_desc}\n")
    za = open(f"{dest}/w035_inklabels.zarr/0/.zarray").read(); open(f"{WORK}/logs/label_zarray.json", "w").write(za)
    print(f"label verificata: file={n} byte={b} tree_sha256={tsha} ({source_desc})"); print(za)
inputs = sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else []
# Il percorso di montaggio dei dataset cambia con il tipo di sessione (6 settembre 2026): /kaggle/input/<slug>/ nel run
# CPU, /kaggle/input/datasets/<utente>/<slug>/ nei run GPU. Nei run seed42 e seed43 il manifest cercato a profondita'
# fissa non veniva trovato e scattava il ripiego (252 s e 1007 s). Si cerca in modo ricorsivo e si prende il manifest
# che sta accanto alla cartella w035_labels.
roots = glob.glob("/kaggle/input/**/w035/w035_inklabels.zarr/0/.zarray", recursive=True)
manifests = [m for m in glob.glob("/kaggle/input/**/manifest.json", recursive=True)
             if os.path.isdir(os.path.join(os.path.dirname(m), "w035_labels"))]
print("montato in /kaggle/input:", inputs, "| manifest:", manifests, "| label:", roots)
if manifests and roots:
    manifest = json.load(open(manifests[0]))
    src = os.path.dirname(os.path.dirname(os.path.dirname(roots[0])))       # .../w035
    files = manifest["files"]
    assert (len(files), sum(f["size"] for f in files)) == EXPECTED, "STOP: manifest del dataset diverso dalla misura del 6 settembre 2026"
    missing = [f["path"] for f in files
               if not (os.path.isfile(os.path.join(src, f["path"][len(PREFIX):])) and os.path.getsize(os.path.join(src, f["path"][len(PREFIX):])) == f["size"])]
    assert not missing, f"STOP: {len(missing)} file della label mancanti o di dimensione diversa, p.es. {missing[:3]}"
    dest = f"{HEAVY}/labels/w035"
    if os.path.isdir(dest):
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    msha = hashlib.sha256(open(manifests[0], "rb").read()).hexdigest()
    check_label_tree(dest, f"kaggle_dataset manifest_sha256={msha}")
    LABEL_READY = True
else:
    print("dataset della label non montato: si passa al download diretto (cella successiva)")


In [ ]:
# Passo 4 (ripiego) — label w035: elenco dall'API del bucket e download diretto in parallelo (~8 min)
import json, os, re, time, urllib.request, concurrent.futures as cf
API = "https://huggingface.co/api/buckets/scrollprize/datasets/tree/ink_9um/labels/native9-scrollprizeorg-21slices/w035"
RESOLVE = "https://huggingface.co/buckets/scrollprize/datasets/resolve/"
PREFIX = "ink_9um/labels/native9-scrollprizeorg-21slices/w035/"
DEST = f"{HEAVY}/labels/w035"
EXPECTED = (5128, 737833)     # misurati via API il 6 settembre 2026

def list_label_files():
    files, url = [], API
    while url:
        req = urllib.request.Request(url, headers={"User-Agent": "papyruslab-e00"})
        with urllib.request.urlopen(req, timeout=60) as r:
            files += [(e["path"], int(e["size"])) for e in json.load(r) if e.get("type") == "file"]
            m = re.search(r'<([^>]+)>;\s*rel="next"', r.headers.get("Link", "") or "")
            url = m.group(1) if m else None
    return files

def fetch(item):
    path, size = item
    assert path.startswith(PREFIX) and ".." not in path, f"STOP: percorso inatteso {path}"
    out = os.path.join(DEST, path[len(PREFIX):])
    if os.path.exists(out) and os.path.getsize(out) == size:
        return size
    os.makedirs(os.path.dirname(out), exist_ok=True)
    last = None
    for attempt in range(8):
        try:
            req = urllib.request.Request(RESOLVE + path, headers={"User-Agent": "papyruslab-e00"})
            with urllib.request.urlopen(req, timeout=60) as r:
                data = r.read()
            if len(data) == size:
                with open(out, "wb") as fh:
                    fh.write(data)
                return size
            last = f"dimensione {len(data)} != {size}"
        except Exception as ex:      # HTTP 429 (rate limit) e 5xx: attesa crescente e nuovo tentativo
            last = ex
        time.sleep(min(60, 5 * 2 ** attempt))
    raise RuntimeError(f"STOP: download fallito per {path}: {last}")

def fallback_download():
    files = list_label_files(); total = sum(s for _, s in files)
    print("elencati", len(files), "file,", total, "byte")
    assert (len(files), total) == EXPECTED, f"STOP: label diversa dalla misura del 6 settembre 2026: {len(files)} file, {total} byte (attesi {EXPECTED})"
    t0 = time.time()
    # Hugging Face limita le richieste anonime parallele (HTTP 429 osservato dal run Kaggle del 6 settembre 2026): pochi thread
    with cf.ThreadPoolExecutor(max_workers=4) as ex:
        got = sum(ex.map(fetch, files))
    print(f"scaricati {got} byte in {time.time() - t0:.0f} s")
    check_label_tree(DEST, "direct_download")     # stesso controllo di contenuto del percorso dataset

if LABEL_READY:
    print("label gia' pronta dal dataset Kaggle: ripiego non necessario")
else:
    fallback_download()


In [ ]:
%%bash
source /kaggle/working/e00/env.sh
disk_check "dopo i download"


In [ ]:
# Passo 5 — prova minima di lettura remota e della label (senza GPU)
import zarr, fsspec, numpy as np, json
root = zarr.open(fsspec.get_mapper(ZARR), mode="r")
a = root["0"]; print(a.shape, a.dtype, a.chunks)
assert tuple(a.shape) == (28, 5820, 5240) and str(a.dtype) == "uint8", "STOP: surface volume con forma o tipo inattesi"
blk = a[6:23, 2816:2944, 2560:2688]; print("blocco", blk.shape, int(blk.min()), int(blk.max()))
assert blk.shape == (17, 128, 128) and int(blk.max()) > 0, "STOP: lettura remota del blocco non valida"
lab = zarr.open(f"{HEAVY}/labels/w035/w035_inklabels.zarr", mode="r")["0"]
msk = zarr.open(f"{HEAVY}/labels/w035/w035_supervision_mask.zarr", mode="r")["0"]
assert tuple(lab.shape) == (28, 5820, 5240) and tuple(msk.shape) == (28, 5820, 5240), "STOP: label o maschera con forma inattesa"
counts = {"n_supervisionati": int((msk[14] > 0).sum()), "n_inchiostro": int(((lab[14] > 0) & (msk[14] > 0)).sum())}
print(counts); json.dump(counts, open(f"{WORK}/logs/label_counts.json", "w"), indent=1)
assert counts["n_supervisionati"] > 0 and counts["n_inchiostro"] > 0, "STOP: label o maschera vuote al piano Z=14"


In [ ]:
# Passo 5 (segue) — prova minima senza GPU: costruire il modello dal checkpoint seed42 esattamente come fa infer.py.
# Eseguita in un SOTTOPROCESSO Python, come l'inferenza vera: il kernel del notebook e' partito prima di `pip install -e`
# e non vede i pacchetti installati in modalita' editable (preflight v6 del 6 settembre 2026).
import subprocess, sys, json
code = r'''
import argparse, json, torch
from koine_machines.inference import infer as kinfer
args = argparse.Namespace(checkpoint="__CKPT__", amp_dtype="auto", model_type="auto", metadata_json=None)
cm = kinfer.configure_model(args)            # carica il payload su CPU e ricostruisce la rete da checkpoint['config']
info = {"in_chans": int(cm.in_chans), "amp_dtype": str(cm.amp_dtype),
        "n_params": int(sum(p.numel() for p in cm.model.parameters())),
        "model_class": type(cm.model).__name__, "preprocessing": str(cm.preprocessing)[:200]}
print("MODEL_BUILD_JSON=" + json.dumps(info))
'''.replace("__CKPT__", f"{HEAVY}/checkpoints/ink_9um/hybrid_3d2d-seed42/step-075000.pth")
r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-1500:])
assert r.returncode == 0, "STOP: costruzione del modello su CPU fallita (vedi stderr sopra)"
info = json.loads(r.stdout.split("MODEL_BUILD_JSON=")[1].splitlines()[0])
json.dump(info, open(f"{WORK}/logs/model_build_cpu.json", "w"), indent=1)
assert info["in_chans"] == 17, "STOP: il checkpoint non dichiara 17 slice in ingresso"
assert "float16" in info["amp_dtype"], f"STOP: AMP dtype inatteso {info['amp_dtype']} (atteso float16 da mixed_precision=fp16)"
print("modello costruito su CPU:", info)

# Prova di scrittura del TIFF con gli stessi parametri di infer.py (uint8, tiled 128x128, LZW), in sottoprocesso
tiff_code = r'''
import numpy as np, tifffile, imagecodecs, json
a = (np.arange(256 * 256) % 251).astype(np.uint8).reshape(256, 256)
tifffile.imwrite("__OUT__", a, dtype=np.uint8, compression="lzw", tile=(128, 128))
b = tifffile.imread("__OUT__")
print("TIFF_TEST_JSON=" + json.dumps({"ok": bool((a == b).all()), "tifffile": tifffile.__version__, "imagecodecs": imagecodecs.__version__}))
'''.replace("__OUT__", f"{WORK}/logs/tiff_lzw_test.tif")
r = subprocess.run([sys.executable, "-c", tiff_code], capture_output=True, text=True)
print(r.stdout[-800:]); print(r.stderr[-800:])
assert r.returncode == 0, "STOP: scrittura TIFF LZW fallita: imagecodecs mancante o incompatibile"
tinfo = json.loads(r.stdout.split("TIFF_TEST_JSON=")[1].splitlines()[0]); assert tinfo["ok"], "STOP: TIFF riletto diverso da quello scritto"
json.dump(tinfo, open(f"{WORK}/logs/tiff_lzw_test.json", "w"), indent=1); print("scrittura TIFF LZW verificata:", tinfo)


In [ ]:
%%bash
# Passo 10 — hash di tutto ciò che viene persistito, stato finale
set -e
source /kaggle/working/e00/env.sh
cp /kaggle/working/e00/env.sh $WORK/logs/env.sh.txt
echo "end=$(date -u +%FT%TZ)" >> $WORK/logs/run_info.txt
disk_check "finale"
cd $WORK && find out logs -type f ! -name SHA256SUMS -print0 | sort -z | xargs -0 sha256sum > out/SHA256SUMS
cat out/SHA256SUMS
echo "persistito: $(du -sh $WORK | cut -f1)"
